<a href="https://colab.research.google.com/github/IgnBys/3D_splatting/blob/main/GS_VTON_with_gaussians.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/yukangcao/GS-VTON.git


fatal: destination path 'GS-VTON' already exists and is not an empty directory.


### Step 1: Install PyTorch 2.2.1 and xformers 0.0.25 (CUDA 11.8)

In [2]:
# Clean up existing conflicting installations first
!pip uninstall -y torch torchvision torchaudio xformers

Found existing installation: torch 2.4.1+cu124
Uninstalling torch-2.4.1+cu124:
  Successfully uninstalled torch-2.4.1+cu124
Found existing installation: torchvision 0.19.1+cu124
Uninstalling torchvision-0.19.1+cu124:
  Successfully uninstalled torchvision-0.19.1+cu124
Found existing installation: torchaudio 2.4.1+cu124
Uninstalling torchaudio-2.4.1+cu124:
  Successfully uninstalled torchaudio-2.4.1+cu124
Found existing installation: xformers 0.0.28.post1
Uninstalling xformers-0.0.28.post1:
  Successfully uninstalled xformers-0.0.28.post1


In [3]:
# Upgrade setuptools to provide the distutils shim required by Python 3.12
!pip install setuptools --upgrade

  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 69.5.1
    Uninstalling setuptools-69.5.1:
      Successfully uninstalled setuptools-69.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
basicsr 1.4.2 requires torch>=1.7, which is not installed.
basicsr 1.4.2 requires torchvision, which is not installed.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
# Устанавливаем PyTorch 2.4.1 (CUDA 12.4) и совместимую с Python 3.12 версию xformers
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu124
!pip install xformers==0.0.28.post1

Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached https://download-r2.pytorch.org/whl/cu124/torch-2.4.1%2Bcu124-cp312-cp312-linux_x86_64.whl (797.1 MB)
ERROR: Operation cancelled by user
^C


### Step 2: Install Additional Dependencies
This includes IP-Adapter, BasicSR, and Detectron2.

In [ ]:
!pip install git+https://github.com/tencent-ailab/IP-Adapter.git

In [ ]:
!pip install git+https://github.com/XPixelGroup/BasicSR@8d56e3a045f9fb3e1d8872f92ee4a4f07f886b0a

In [ ]:
import os

# 1. Используем системную CUDA по умолчанию
os.environ['CUDA_HOME'] = '/usr/local/cuda'

# 2. Clone and Patch Detectron2 source
%cd /content
!rm -rf detectron2
!git clone https://github.com/facebookresearch/detectron2.git
%cd detectron2
!find . -name "*.h" -o -name "*.cpp" -o -name "*.cu" | xargs sed -i '1i #include <cstdint>'

# 3. Install build dependencies
!pip install -q fvcore iopath yacs

# 4. Build from the patched source (с использованием текущего окружения PyTorch)
!python -m pip install . --no-build-isolation

In [ ]:
import detectron2

### Step 3: Install GS-VTON Requirements
Navigating into the cloned directory and installing the `requirements.txt`.

In [ ]:
%cd /content/GS-VTON

In [ ]:
# 1. Обновляем сабмодули
!git submodule update --init --recursive

In [ ]:
# 2. Скачиваем и устанавливаем nvdiffrast
!git clone https://github.com/NVlabs/nvdiffrast.git /content/nvdiffrast
%cd /content/nvdiffrast
!pip install . --no-build-isolation

In [ ]:
import sys

In [ ]:
!pip install -q plyfile ninja

In [ ]:
!{sys.executable} -m pip install /content/GS-VTON/stage2/gaussiansplatting/submodules/diff-gaussian-rasterization --no-build-isolation

In [ ]:
!sed -i '1i #include <float.h>' /content/GS-VTON/stage2/gaussiansplatting/submodules/simple-knn/simple_knn.cu
!{sys.executable} -m pip install /content/GS-VTON/stage2/gaussiansplatting/submodules/simple-knn --no-build-isolation

In [ ]:
# 4. Чистим requirements и ставим оставшиеся зависимости
!sed -i '/nvdiffrast/d' requirements.txt
!sed -i '/diff-gaussian-rasterization/d' requirements.txt
!sed -i '/simple-knn/d' requirements.txt
!sed -i '/tiny-cuda-nn/d' requirements.txt
!pip install -r requirements.txt

In [ ]:
# 5. Отдельно ставим tiny-cuda-nn без изоляции сборки
!pip install ninja
!pip install git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch --no-build-isolation

In [ ]:
# Понижаем версию setuptools для совместимости с Python 3.12 при сборке
!pip install "setuptools<70.0.0"

In [ ]:
import os
%cd /content
!rm -rf tiny-cuda-nn
!git clone --recursive https://github.com/NVlabs/tiny-cuda-nn
%cd tiny-cuda-nn/bindings/torch

# Устанавливаем tiny-cuda-nn
!pip install . --no-build-isolation

In [ ]:
# Репозиторий gaussian-splatting (submodules) уже находится внутри структуры GS-VTON.

In [ ]:
import diff_gaussian_rasterization
import simple_knn

In [ ]:
import os

# Definiowanie ścieżek
base_path = '/content/GS-VTON/stage1/ckpt'
folders = [
    'densepose',
    'openpose/ckpts',
    'humanparsing'
]

# Tworzenie folderów
for folder in folders:
    path = os.path.join(base_path, folder)
    os.makedirs(path, exist_ok=True)
    print(f"Utworzono: {path}")

In [ ]:
# Definiowanie linków (zmienione na wersje raw/resolve dla Hugging Face)
models = {
    '/content/GS-VTON/stage1/ckpt/densepose/model_final_162be9.pkl': 'https://huggingface.co/yisol/IDM-VTON/resolve/main/densepose/model_final_162be9.pkl',
    '/content/GS-VTON/stage1/ckpt/openpose/ckpts/body_pose_model.pth': 'https://huggingface.co/yisol/IDM-VTON/resolve/main/openpose/ckpts/body_pose_model.pth',
    '/content/GS-VTON/stage1/ckpt/humanparsing/parsing_atr.onnx': 'https://huggingface.co/yisol/IDM-VTON/resolve/main/humanparsing/parsing_atr.onnx',
    '/content/GS-VTON/stage1/ckpt/humanparsing/parsing_lip.onnx': 'https://huggingface.co/yisol/IDM-VTON/resolve/main/humanparsing/parsing_lip.onnx'
}

# Pobieranie plików
for path, url in models.items():
    if not os.path.exists(path):
        print(f'Pobieranie do: {path}')
        !wget -O "{path}" "{url}"
    else:
        print(f'Plik już istnieje: {path}')

print('\nWszystkie pliki zostały pobrane.')

In [ ]:
# 1. Tworzenie struktury dla Stage 2
stage2_path = '/content/GS-VTON/stage2/Self_Correction_Human_Parsing'
os.makedirs(stage2_path, exist_ok=True)
print(f"Utworzono folder: {stage2_path}")


In [ ]:
# New direct download attempt for OneDrive
url = 'https://entuedu-my.sharepoint.com/:u:/g/personal/yukang_cao_staff_main_ntu_edu_sg/EWhlfuAFDnhAmB3WliRnqxsBCWT6q9-n97wi82czlxzrAg?download=1'
target = '/content/GS-VTON/stage2/Self_Correction_Human_Parsing/logits.pt'

print(f"Trying to download logits.pt from: {url}")
# -L follows redirects, -c resumes, -O specifies output
!wget -L -O "{target}" "{url}"

if os.path.exists(target):
    size = os.path.getsize(target)
    if size > 1000000:
        print(f"\n✅ Success! File size: {size / (1024*1024):.2f} MB")
    else:
        print("\n❌ Failed: The downloaded file is too small. OneDrive is still blocking direct access.")
        print("Please download it in your browser and upload manually.")

#Połączenie z gaussian-splatting i pobranie niezbędnych danych do

In [ ]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile ninja

In [ ]:
%cd /content/gaussian-splatting

In [ ]:
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization

In [ ]:
!nvcc --version

import torch
print("PyTorch version:", torch.__version__)
print("PyTorch CUDA version:", torch.version.cuda)

In [ ]:
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'

In [ ]:
# The compiler is complaining that FLT_MAX is undefined.
# In newer versions of CUDA and GCC, certain headers like <float.h> are no longer included automatically,
# so we manually insert the missing header into the source code and then reinstall simple_knn.
!sed -i '1i #include <float.h>' /content/gaussian-splatting/submodules/simple-knn/simple_knn.cu

%cd /content/gaussian-splatting
!pip install ./submodules/simple-knn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# After mounting, define your image path
# Example: IMAGE_PATH = '/content/drive/MyDrive/magisterka/ignat_photos'
import os
print("Google Drive mounted successfully.")

In [ ]:
import os
%cd /content/gaussian-splatting
print(f"Current working directory: {os.getcwd()}")

In [ ]:
import os
import shutil
import glob

# Define the paths
SOURCE_PHOTOS = '/content/drive/MyDrive/magisterka/ignat_photos'
REPO_DATA_PATH = '/content/gaussian-splatting/data/ignat_photos'
REPO_INPUT_PATH = os.path.join(REPO_DATA_PATH, 'input')

# 1. Clear and recreate the input directory to ensure a clean state
if os.path.exists(REPO_INPUT_PATH):
    shutil.rmtree(REPO_INPUT_PATH)
os.makedirs(REPO_INPUT_PATH, exist_ok=True)

# 2. Copy all photos recursively from Drive to the local data/input folder
print(f"Searching for photos in {SOURCE_PHOTOS}...")
extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
found_files = []
for ext in extensions:
    found_files.extend(glob.glob(os.path.join(SOURCE_PHOTOS, '**', ext), recursive=True))

print(f"Copying {len(found_files)} photos to {REPO_INPUT_PATH}...")
for i, file_path in enumerate(found_files):
    # Flatten the structure by using just the filename
    dest_path = os.path.join(REPO_INPUT_PATH, os.path.basename(file_path))
    shutil.copy(file_path, dest_path)



In [ ]:
SOURCE_PATH = '/content/drive/MyDrive/magisterka/ignat_photos'

if os.path.exists(SOURCE_PATH):
    photos = os.listdir(SOURCE_PATH)
    print(f'Successfully found folder with {len(photos)} items.')
else:
    print(f'Error: The path {SOURCE_PATH} does not exist. Please check the folder name.')

In [ ]:
# All of these is to use GPU-accelerated version of COLMAP

# Install xvfb for OpenGL virtual display and required dependencies to build COLMAP
!sudo apt-get update
!sudo apt-get install -y xvfb cmake ninja-build \
    libboost-program-options-dev libboost-filesystem-dev libboost-graph-dev \
    libboost-system-dev libboost-test-dev libsuitesparse-dev \
    libfreeimage-dev libgoogle-glog-dev libgflags-dev \
    libglew-dev libqt5opengl5-dev qt5-qmake qtbase5-dev libceres-dev \
    libflann-dev libcgal-dev libmetis-dev

# Build COLMAP from source for GPU (CUDA) support
!rm -rf /content/colmap
!git clone --branch 3.9 https://github.com/colmap/colmap.git /content/colmap
%cd /content/colmap
!mkdir build
%cd build
!cmake .. -GNinja -DCUDA_ENABLED=ON -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=native
!ninja
!sudo ninja install

# Return to the gaussian-splatting directory
%cd /content/gaussian-splatting


In [ ]:
# Run COLMAP conversion using xvfb-run to allow OpenGL context creation
!xvfb-run python convert.py -s /content/gaussian-splatting/data/ignat_photos

In [ ]:
!python train.py -s /content/gaussian-splatting/data/ignat_photos/